# Trabalho Final: Redes Neurais Profundas - **Etapa 4**
**Universidade Federal de Goias (UFG) - Instituto de Informatica (INF)**

**Projeto:** Analise Arquitetural do Congelamento de Camadas na Mitigacao de *Domain Shift* e *Language Shift* em Transformers Multilingues

**Equipe:** Geovana Teixeira Camargo, Sebastiao Correa Fraga Neto, Pedro Reis Pimenta, Henrique Matheus Mendonca de Miranda

---
> **Status:** **Etapa 4 - Analise Estatistica e Visualizacao.** Este notebook consome o `results.csv` (48 linhas) gerado na Etapa 3 e produz a tabela consolidada de F1-macro, os *deltas* de Domain/Language Shift, os testes de significancia (Welch *t* + Mann-Whitney U + Cohen's *d*), o veredito formal das hipoteses **H1**/**H2** e os graficos (heatmap + barplot). **Nao precisa de GPU.**


---
## Etapa 4 - Analise e Graficos

| Fase | Descricao | Status |
|------|-----------|--------|
| **4.0** | Setup: imports + carga do `results.csv` + VERIFY (48x7) | OK |
| **4.1** | Tabela de agregacao: F1-macro media +/- desvio (3 seeds) por config x teste | OK |
| **4.2** | Delta-shift + testes estatisticos vs. baseline C1 + veredito H1/H2 | OK |
| **4.3** | Visualizacoes: heatmap config x teste e barplot dos Delta-shift | OK |

**Mapeamento dos cenarios** (design 2x2; treino sempre em S1 = EN/Eletronicos):

| Teste | Conjunto | Tipo de shift |
|-------|----------|---------------|
| **T1** | S1_val - EN/Eletronicos | nenhum (baseline in-domain, in-language) |
| **T2** | S2 - EN/Beleza | **Domain Shift** |
| **T3** | S3 - PT/Eletronicos | **Language Shift** |
| **T4** | S4 - PT/Beleza | Domain + Language |


### 4.0 - Setup e ingestao de dados

- **INPUT:** `results.csv` (raiz do repositorio, gerado na Etapa 3).
- **ACOES:** clonar o repo (no Colab), importar `pandas`/`numpy`/`matplotlib`/`seaborn`/`scipy`, carregar o CSV, tema `whitegrid`.
- **VERIFY:** `DataFrame` com **48 linhas x 7 colunas**; 4 configs, 3 seeds, 4 testes.


In [ ]:
# No Google Colab, descomente para clonar o repositorio (traz results.csv + src/):
# !git clone https://github.com/Ricktheus/Trabalho-de-Redes-Neurais-Profundas.git
# %cd Trabalho-de-Redes-Neurais-Profundas

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='talk')

df = pd.read_csv('results.csv')

ORDEM_C = ['C1', 'C2', 'C3', 'C4']
ORDEM_T = ['T1', 'T2', 'T3', 'T4']
NOME_CONFIG = {
    'C1': 'C1 - Full fine-tuning',
    'C2': 'C2 - Freeze Lower (0-5)',
    'C3': 'C3 - Freeze Upper (6-11)',
    'C4': 'C4 - Frozen Encoder',
}
NOME_TESTE = {
    'T1': 'T1 - EN/Elec (baseline)',
    'T2': 'T2 - EN/Beleza (Dominio)',
    'T3': 'T3 - PT/Elec (Lingua)',
    'T4': 'T4 - PT/Beleza (Ambos)',
}

# VERIFY
assert df.shape == (48, 7), f'Esperado (48, 7), obtido {df.shape}'
assert sorted(df.config.unique()) == ORDEM_C
assert sorted(df.teste.unique()) == ORDEM_T
assert sorted(df.seed.unique()) == [42, 123, 2024]
print('VERIFY OK -', df.shape, '| configs:', ORDEM_C, '| seeds: [42, 123, 2024]')
df.head()

### 4.1 - Tabela de agregacao (F1-macro: media +/- desvio sobre as 3 seeds)

Formato pivot (4 configs x 4 testes) pronto para o *paper*.


In [ ]:
media  = df.pivot_table(index='config', columns='teste', values='f1_macro', aggfunc='mean').loc[ORDEM_C, ORDEM_T]
desvio = df.pivot_table(index='config', columns='teste', values='f1_macro', aggfunc='std').loc[ORDEM_C, ORDEM_T]

tabela = media.copy().astype(object)
for c in ORDEM_C:
    for t in ORDEM_T:
        tabela.loc[c, t] = f'{media.loc[c, t]:.4f} +/- {desvio.loc[c, t]:.4f}'
print('F1-macro - media +/- desvio (3 seeds)')
tabela

### 4.2 - Delta-shift, testes estatisticos e veredito das hipoteses

- **Delta-shift** (em pontos percentuais, pp): `F1(T1) - F1(Tx)`. Valor **> 0 = perda**; **< 0 = ganho zero-shot**.
- **Testes** vs. baseline **C1** (n=3 seeds/grupo): Welch *t* (variancias desiguais), Mann-Whitney U e **Cohen's d**.
- **Criterio de confirmacao** (Metodologia): mitigacao real exige **Delta >= +3 pp E p < 0.10**.


In [ ]:
# Delta-shift por config
linhas = []
for c in ORDEM_C:
    t1, t2, t3, t4 = (media.loc[c, t] for t in ORDEM_T)
    linhas.append({'config': c, 'F1_T1': round(t1, 4),
                   'D_dominio_T1-T2_pp': round((t1 - t2) * 100, 3),
                   'D_lingua_T1-T3_pp':  round((t1 - t3) * 100, 3),
                   'D_ambos_T1-T4_pp':   round((t1 - t4) * 100, 3)})
deltas = pd.DataFrame(linhas).set_index('config')
print('Delta-shift (pp) - >0 = perda, <0 = ganho zero-shot')
deltas

In [ ]:
def cohen_d(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    na, nb = len(a), len(b)
    sp2 = ((na-1)*a.var(ddof=1) + (nb-1)*b.var(ddof=1)) / (na+nb-2)
    return 0.0 if sp2 == 0 else (a.mean()-b.mean())/np.sqrt(sp2)

def amostras(config, teste):
    return df[(df.config==config)&(df.teste==teste)].sort_values('seed')['f1_macro'].to_numpy()

comparacoes = [
    ('T2', 'C2', 'Domain - Freeze Lower vs Full'),
    ('T2', 'C3', 'Domain - Freeze Upper vs Full (aposta H2)'),
    ('T2', 'C4', 'Domain - Frozen Encoder vs Full'),
    ('T3', 'C2', 'Language - Freeze Lower vs Full (aposta H1)'),
    ('T3', 'C3', 'Language - Freeze Upper vs Full'),
    ('T3', 'C4', 'Language - Frozen Encoder vs Full'),
]
rows = []
for teste, cfg, desc in comparacoes:
    base, alt = amostras('C1', teste), amostras(cfg, teste)
    t_stat, p_t = stats.ttest_ind(alt, base, equal_var=False)
    u_stat, p_u = stats.mannwhitneyu(alt, base, alternative='two-sided')
    rows.append({'cenario': teste, 'comp': f'{cfg} vs C1', 'descricao': desc,
                 'D_pp': round((alt.mean()-base.mean())*100, 3),
                 'p_welch': round(float(p_t), 4), 'p_MWU': round(float(p_u), 4),
                 'cohen_d': round(float(cohen_d(alt, base)), 3),
                 'sig_p<0.10': 'SIM' if p_t < 0.10 else 'nao'})
estat = pd.DataFrame(rows)
estat

In [ ]:
# Veredito formal H1 / H2 (criterio: Delta >= +3 pp E p < 0.10)
def veredito(cfg_alt, teste, nome, pre_t1, pre_tx):
    alt, base = amostras(cfg_alt, teste), amostras('C1', teste)
    d = (alt.mean()-base.mean())*100
    _, p = stats.ttest_ind(alt, base, equal_var=False)
    shift = (media.loc['C1', pre_t1] - media.loc['C1', pre_tx])*100
    ok = (d >= 3.0) and (p < 0.10)
    print(nome)
    print(f'  Pre-condicao (shift na baseline C1, {pre_t1}-{pre_tx}) = {shift:+.2f} pp')
    print(f'  Delta F1({cfg_alt},{teste}) - F1(C1,{teste}) = {d:+.2f} pp | p = {p:.4f}')
    print(f'  -> VEREDITO: {"CONFIRMADA" if ok else "REFUTADA"}\n')

veredito('C2', 'T3', 'H1 (Freeze Lower mitiga Language Shift):', 'T1', 'T3')
veredito('C3', 'T2', 'H2 (Freeze Upper mitiga Domain Shift):',  'T1', 'T2')

# Achado emergente: C2 (Freeze Lower) protege contra Domain Shift
alt, base = amostras('C2', 'T2'), amostras('C1', 'T2')
d = (alt.mean()-base.mean())*100
_, p = stats.ttest_ind(alt, base, equal_var=False)
print('ACHADO EMERGENTE - Freeze Lower (C2) vs Full (C1) em Domain Shift (T2):')
print(f'  Delta = {d:+.2f} pp | p = {p:.4f} -> {"SIGNIFICATIVO" if p < 0.10 else "n.s."} a p<0.10')

### 4.3 - Visualizacoes

**(1) Heatmap** F1-macro cruzando as 4 configuracoes x 4 cenarios.


In [ ]:
hm = media.copy()
hm.index   = [NOME_CONFIG[c] for c in hm.index]
hm.columns = [NOME_TESTE[t] for t in hm.columns]
plt.figure(figsize=(11, 7))
ax = sns.heatmap(hm, annot=True, fmt='.3f', cmap='viridis', linewidths=0.5,
                 cbar_kws={'label': 'F1-macro (media de 3 seeds)'})
ax.set_title('F1-macro por Configuracao x Cenario de Teste', pad=14, weight='bold')
ax.set_xlabel(''); ax.set_ylabel('')
plt.xticks(rotation=25, ha='right'); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

**(2) Barplot** dos Delta-shift (Dominio T1-T2 vs. Lingua T1-T3) com barras de erro (desvio entre seeds).


In [ ]:
def deltas_por_seed(config, ta, tb):
    a = df[(df.config==config)&(df.teste==ta)].sort_values('seed')['f1_macro'].to_numpy()
    b = df[(df.config==config)&(df.teste==tb)].sort_values('seed')['f1_macro'].to_numpy()
    return (a - b) * 100

dom_m=[]; dom_s=[]; lin_m=[]; lin_s=[]
for c in ORDEM_C:
    dd = deltas_por_seed(c, 'T1', 'T2'); dl = deltas_por_seed(c, 'T1', 'T3')
    dom_m.append(dd.mean()); dom_s.append(dd.std(ddof=1))
    lin_m.append(dl.mean()); lin_s.append(dl.std(ddof=1))

x = np.arange(len(ORDEM_C)); w = 0.38
plt.figure(figsize=(11, 7))
plt.bar(x - w/2, dom_m, w, yerr=dom_s, capsize=5, label='Domain Shift (T1-T2)', color='#d1495b')
plt.bar(x + w/2, lin_m, w, yerr=lin_s, capsize=5, label='Language Shift (T1-T3)', color='#30638e')
plt.axhline(0, color='black', lw=0.8)
plt.xticks(x, ORDEM_C); plt.ylabel('Delta F1-macro vs. baseline T1 (pp)')
plt.title('Queda de F1-macro sob Domain e Language Shift\n(>0 = perda; <0 = ganho zero-shot)', weight='bold')
plt.legend(); plt.tight_layout(); plt.show()

---
## Conclusao da Etapa 4

1. **H1 (Language Shift) - REFUTADA.** Nao ha *Language Shift* a mitigar: o modelo e **melhor** em PT (T3) do que em EN (T1) sem ter treinado em PT (ganho zero-shot). C2 vs C1 em T3: Delta ~ +0.03 pp (p ~ 0.93, n.s.).
2. **H2 (Domain Shift) - REFUTADA e invertida.** Congelar o topo (C3) **piorou** o *Domain Shift* (Delta ~ -2.97 pp, p ~ 0.07).
3. **Achado emergente.** Quem mitiga o *Domain Shift* e **C2 (Freeze Lower)**: Delta ~ +1.19 pp sobre C1 em T2 (p ~ 0.09) - as representacoes multilingues iniciais do XLM-R atuam como **regularizador** contra o vies de dominio.
4. **C4 (Frozen Encoder)** colapsa em todos os cenarios - o *probing* linear nao basta para a tarefa.
